In [61]:
"""
This calculator was made by Lennart Baumgärtel, last edited 4.1.25. If you encounter any bugs, send me a message or mail at 202204791@post.au.dk.
Check that you have the newest version by comparing it to the file at https://drive.google.com/drive/folders/1_QNQATyqTWfCe3uoIofKUVyg2fKdOrod?usp=sharing and checking the changelog
Feel free to share with whomever, get this out to people and get it used. Good luck!
Current Version: 1.2.1 (1.2 is also a stable version)
Bugs encountered: 2
"""
import numpy as np
import scipy.constants as u
#######
#Constants
#######
c = u.c  # speed of light (m/s)
hbar = u.hbar  # reduced Planck's constant (Js)
e = u.elementary_charge  # elementary charge (C)
e_0 = u.epsilon_0  # permittivity of free space (F/m)
m_n = u.neutron_mass  # neutron mass (kg)
m_p = u.proton_mass # proton mass (kg)
m_e = u.electron_mass # electron mass (kg)
Kg_to_MeV = 5.60958616721986e29
MeV_to_Kg=1.79e-30
KeV_to_Kg=1.79e-33

#######
#Services
#######

def SEMF(A,Z,units= "MeV"):

    # Constants in MeV / c**2
    a_v = 15.75     # Volume term
    a_s = 17.8      # Surface term
    a_c = 0.711     # Coulomb term
    a_sym = 23.7    # Asymmetry term   
    a_pair = 11.2   # Pairing Term constant
   
    # Defining the masses of the particles into Mev/c^2
    m_n = u.neutron_mass * Kg_to_MeV
    m_p = u.proton_mass * Kg_to_MeV
    m_e = u.electron_mass * Kg_to_MeV

    # Calculating the pairing term,
    def f5(A,Z):
        if A%2 ==0 and Z%2 == 0: # If both A and Z are even
            return a_pair*A**(-1/2)
        if A%2 == 1 and Z%2 ==1: # If both A and Z are uneven
            return -a_pair*A**(-1/2)
        else:
            return 0 # One and only one is even
    Binding_energy = ( a_v * A
                    - a_s * A ** (2/3)
                    - a_c * Z * (Z - 1) * A ** (-1/3)
                    - a_sym * (Z - (A / 2) )**2 / A
                    + f5(A,Z)
                    ) 
    mass = Z * (m_p + m_e) + (A-Z) * m_n - Binding_energy
    # Unit conversion
    if units == "MeV":
        return mass  # Mass in MeV/c^2
    elif units == "Kg":
        return mass * 1.78266e-30  # Mass in kilograms
    elif units == "AU":
        return mass / 931.494  # Mass in atomic mass units (u)
    else:
        raise ValueError("Unit needs to be either MeV, Kg, or AU") 



def halflife(A,Z,Q,V0,a=False, units = "MeV", timeframe = "Seconds"):
    # units refers to the unitsystem that the energy is given in. Can be either Joule or MeV. If it is in any other unit, please convert before using
    # timeframe refers to the timeframe in which the result shall be given. Can be either Seconds, Hours or Years
    # A is the number of protons and neutrons
    # Z is the number of protons
    # Q is the amount of energy that can be freed by performing a alpha decay
    # V0 is the binding energy of the given nucleus
    
    #Conversion into SI

    if units == "Joule":
        # 1 J = 6.24150907e18 eV
        Q *= 6.24150907e18 * 1e-6  # results in the energy being in MeV
        V0 *= 6.24150907e18 * 1e-6 # results in the energy being in MeV
    elif units == "MeV":
        pass
    else:
        raise ValueError("Energy neds to be either in MeV or Joule")
    
    # Constants
    m = 3727.379 #MeV This is the mass of 2 neutrons + 2 protons (mass of alpha particle) in NU
    alfa = 1 / 137 # finestructure constant
    if a == False:
        a = 1.35 * A**(1 / 3) # An approximation for the radius of the nucleus 
    B = alfa * 2 *c*hbar* (Z-2) / a # Potential energy Barrier B (Terranova 11.38)

    frontfactor = (0.693 * a # e23 comes from e8 * e15, as the speed of light is in e8 meters/second, but a is given in femtometer (e15). Therefore, we need to convert speed of light into femtometers/second, which is 2.99e23
            * np.sqrt(m /(2*(V0 + Q))))

    expfac = 2 * np.sqrt(2 * m / Q) * 2 * (Z-2) * c*hbar* alfa * (np.pi / 2 - 2 * np.sqrt(Q / B))
    timeconversion = 1 
    if timeframe == "Seconds":
        pass
    elif timeframe == "Hours":
        timeconversion = 3600
    elif timeframe == "Years":
        timeconversion =  (365.25 * 24 * 3600)
    else:
        raise ValueError("Timeframe should be either Seconds, Hours or Years")

    result = frontfactor * np.exp(expfac) / timeconversion 
    return result

def compton(m,unit = "kg"): 
    # m is the mass of the particles, whos compton wavelength shall be calculated
    if unit == "kg": # In this case, the resulting Wavelength will we in meters
        return (2*np.pi*hbar) / (m * c)
    elif unit == "keV" or unit == "MeV": # In this case, the wavelength will be in inverse keV or inverse MeV, depending on the input mass
        return (2*np.pi) / (m)
    else:
        raise ValueError("unit needs to be either kg, keV or Mev")
    
print(halflife(228,88,4,40,8))



36.08142344172796
